In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D12 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D12 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path

import hashlib
import json
import re
import unicodedata

import pandas as pd

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D12"
DOCUMENT_NAME = (
    "Our World in Data — Annual CO2 emissions time series"
)

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_RECORD_COUNT = 25

REFERENCE_CATEGORY = (
    "Environmental time-series"
)

REFERENCE_TOPIC = (
    "Annual CO2 emissions"
)

REFERENCE_DESCRIPTION = (
    "Annual CO2 emissions"
)

REFERENCE_UNIT = None


TARGET_YEARS = [
    1750,
    1800,
    1850,
    1900,
    1950,
    1960,
    1970,
    1980,
    1990,
    2000,
    2010,
    2011,
    2012,
    2013,
    2014,
    2015,
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
]


EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]


ALIGNMENT_IDENTITY_FIELDS = [
    "Reporting Period"
]


PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Source Location"
]


EXPECTED_CATEGORY_COUNTS = {
    REFERENCE_CATEGORY:
        EXPECTED_RECORD_COUNT
}


OUTPUT_DIR = Path(
    "outputs_D12_validation_branch_B"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH, "-", BRANCH_NAME)
print(
    "Alignment identity fields:",
    ALIGNMENT_IDENTITY_FIELDS
)
print(
    "Primary correctness fields:",
    PRIMARY_CORRECTNESS_FIELDS
)

In [ ]:
# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D12_reference_values.csv
#   2) D12_branch_B_parsed_extraction.json
#   3) D12_branch_B_technical_diagnostics.json

uploaded = files.upload()

uploaded_files = list(
    uploaded.keys()
)


csv_files = [
    file_name
    for file_name in uploaded_files
    if file_name.lower().endswith(
        ".csv"
    )
]


json_files = [
    file_name
    for file_name in uploaded_files
    if file_name.lower().endswith(
        ".json"
    )
]


if len(csv_files) != 1:

    raise ValueError(
        "Upload exactly one D12 "
        "Stage 1 reference CSV."
    )


if len(json_files) != 2:

    raise ValueError(
        "Upload exactly two JSON files: "
        "the Branch B parsed extraction "
        "and technical diagnostics."
    )


REFERENCE_FILE = (
    csv_files[0]
)

PARSED_EXTRACTION_FILE = None

TECHNICAL_DIAGNOSTICS_FILE = None


for file_name in json_files:

    with open(
        file_name,
        "r",
        encoding="utf-8-sig"
    ) as f:

        obj = json.load(f)


    if not isinstance(
        obj,
        dict
    ):
        continue


    if (
        obj.get(
            "document_id"
        ) == DOCUMENT_ID

        and obj.get(
            "branch"
        ) == BRANCH

        and isinstance(
            obj.get(
                "records"
            ),
            list
        )
    ):

        PARSED_EXTRACTION_FILE = (
            file_name
        )


    if (
        obj.get(
            "document_id"
        ) == DOCUMENT_ID

        and obj.get(
            "branch"
        ) == BRANCH

        and "structurally_evaluable"
        in obj

        and "record_schema_valid"
        in obj

        and "valid_json"
        in obj
    ):

        TECHNICAL_DIAGNOSTICS_FILE = (
            file_name
        )


if PARSED_EXTRACTION_FILE is None:

    raise ValueError(
        "Could not identify the D12 "
        "Branch B parsed extraction."
    )


if TECHNICAL_DIAGNOSTICS_FILE is None:

    raise ValueError(
        "Could not identify the D12 "
        "Branch B technical diagnostics."
    )


print(
    "Reference:",
    REFERENCE_FILE
)

print(
    "Parsed extraction:",
    PARSED_EXTRACTION_FILE
)

print(
    "Technical diagnostics:",
    TECHNICAL_DIAGNOSTICS_FILE
)

In [ ]:
# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)


schema_validity = bool(
    structurally_evaluable
)


schema_diagnostics = {

    "valid_json":
        bool(
            technical_diagnostics.get(
                "valid_json",
                False
            )
        ),

    "record_schema_valid":
        bool(
            technical_diagnostics.get(
                "record_schema_valid",
                False
            )
        ),

    "field_types_valid":
        bool(
            technical_diagnostics.get(
                "field_types_valid",
                False
            )
        ),

    "structurally_evaluable":
        structurally_evaluable,

    "schema_validity":
        schema_validity
}


if not structurally_evaluable:

    raise ValueError(
        "D12 Branch B output is not "
        "structurally evaluable. "
        "Content-level validation "
        "cannot proceed."
    )


print(
    json.dumps(
        schema_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 5. Confirm fixed Stage 1 reference semantics
# ============================================================

reference_schema_exact = (
    reference_df.columns.tolist()
    == EXPECTED_FIELDS
)


reference_record_count_valid = (
    len(
        reference_df
    )
    == EXPECTED_RECORD_COUNT
)


reference_category_counts = (
    reference_df[
        "Category"
    ]
    .value_counts()
    .to_dict()
)


reference_category_counts_valid = (
    reference_category_counts
    == EXPECTED_CATEGORY_COUNTS
)


reference_topic_constant = bool(
    (
        reference_df[
            "Topic"
        ]
        == REFERENCE_TOPIC
    ).all()
)


reference_description_constant = bool(
    (
        reference_df[
            "Description"
        ]
        == REFERENCE_DESCRIPTION
    ).all()
)


reference_unit_null = bool(

    reference_df[
        "Unit"
    ].apply(

        lambda value:

            value is None

            or (
                isinstance(
                    value,
                    str
                )
                and value.strip() == ""
            )

            or pd.isna(
                value
            )

    ).all()
)


reference_values_numeric = (
    pd.to_numeric(
        reference_df[
            "Value"
        ],
        errors="coerce"
    )
    .notna()
    .all()
)


reference_values_non_negative = bool(

    (
        pd.to_numeric(
            reference_df[
                "Value"
            ],
            errors="coerce"
        )
        >= 0
    ).all()
)


reference_periods = (
    pd.to_numeric(
        reference_df[
            "Reporting Period"
        ],
        errors="coerce"
    )
)


reference_years_valid = (

    reference_periods
    .notna()
    .all()

    and

    reference_periods
    .astype(int)
    .tolist()
    == TARGET_YEARS
)


reference_identity_unique = (

    reference_df[
        "Reporting Period"
    ]
    .astype(str)
    .nunique()
    == EXPECTED_RECORD_COUNT
)


reference_source_locations_valid = bool(

    reference_df[
        "Source Location"
    ]
    .astype(str)
    .str.fullmatch(
        r"CSV data row \d+"
    )
    .all()
)


reference_semantic_checks = {

    "reference_schema_exact":
        bool(
            reference_schema_exact
        ),

    "reference_record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "reference_category_counts_valid":
        bool(
            reference_category_counts_valid
        ),

    "reference_topic_constant":
        bool(
            reference_topic_constant
        ),

    "reference_description_constant":
        bool(
            reference_description_constant
        ),

    "reference_unit_null":
        bool(
            reference_unit_null
        ),

    "reference_values_numeric":
        bool(
            reference_values_numeric
        ),

    "reference_values_non_negative":
        bool(
            reference_values_non_negative
        ),

    "reference_years_valid":
        bool(
            reference_years_valid
        ),

    "reference_identity_unique":
        bool(
            reference_identity_unique
        ),

    "reference_source_locations_valid":
        bool(
            reference_source_locations_valid
        )
}


reference_semantics_valid = all(
    reference_semantic_checks.values()
)


if not reference_semantics_valid:

    raise AssertionError(
        "D12 Stage 1 reference semantics "
        "are not valid."
    )


print(
    json.dumps(
        reference_semantic_checks,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 6. Comparison-only canonicalisation
# ============================================================


def normalise_text(value):

    if value is None:
        return None


    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )


    text = (
        text
        .replace(
            "\u00a0",
            " "
        )
        .replace(
            "\u2007",
            " "
        )
        .replace(
            "\u202f",
            " "
        )
        .replace(
            "—",
            "-"
        )
        .replace(
            "–",
            "-"
        )
        .replace(
            "’",
            "'"
        )
    )


    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def canonical_period(value):

    text = normalise_text(
        value
    )


    if text is None:
        return None


    if re.fullmatch(
        r"\d{4}",
        text
    ):

        return int(
            text
        )


    return text


def parse_numeric(value):

    if (
        value is None
        or isinstance(
            value,
            bool
        )
    ):

        return None


    if isinstance(
        value,
        (int, float)
    ):

        return float(
            value
        )


    text = (
        str(value)
        .strip()
        .replace(
            ",",
            ""
        )
    )


    if re.fullmatch(
        r"[-+]?\d+(?:\.\d+)?",
        text
    ):

        return float(
            text
        )


    return None


def canonical_null(value):

    if value is None:
        return None


    if (
        isinstance(
            value,
            float
        )
        and pd.isna(
            value
        )
    ):

        return None


    if (
        isinstance(
            value,
            str
        )
        and value.strip() == ""
    ):

        return None


    return normalise_text(
        value
    )


def canonical_source_location(value):

    text = normalise_text(
        value
    )


    if text is None:
        return None


    match = re.fullmatch(
        r"CSV data row (\d+)",
        text
    )


    return (
        f"CSV data row "
        f"{int(match.group(1))}"

        if match

        else text
    )

In [ ]:
# ============================================================
# 7. Deterministic one-to-one alignment by Reporting Period
# ============================================================


def build_identity_index(
    records,
    dataset_name
):

    index = {}

    duplicate_groups = []


    for (
        record_index,
        record
    ) in enumerate(
        records
    ):

        if not isinstance(
            record,
            dict
        ):
            continue


        identity = canonical_period(
            record.get(
                "Reporting Period"
            )
        )


        if identity in index:

            duplicate_groups.append({
                "dataset":
                    dataset_name,

                "identity":
                    identity,

                "record_index":
                    record_index
            })

        else:

            index[
                identity
            ] = {

                "record_index":
                    record_index,

                "record":
                    record
            }


    return (
        index,
        duplicate_groups
    )


reference_records = (
    reference_df
    .to_dict(
        "records"
    )
)


(
    reference_index,
    reference_duplicates

) = build_identity_index(

    reference_records,
    "Reference"
)


(
    extraction_index,
    extraction_duplicates

) = build_identity_index(

    extracted_records,
    "Extraction"
)


alignment_issues = (
    reference_duplicates
    + extraction_duplicates
)


print(
    "Reference duplicate identities:",
    len(
        reference_duplicates
    )
)

print(
    "Extraction duplicate identities:",
    len(
        extraction_duplicates
    )
)


if reference_duplicates:

    raise AssertionError(
        "Reference Reporting Period "
        "identity is not unique."
    )


if extraction_duplicates:

    raise AssertionError(
        "Extraction Reporting Period "
        "identity is not unique."
    )

In [ ]:
# ============================================================
# 8. Match records and compare fields
# ============================================================


def compare_field(
    field,
    reference_value,
    extracted_value
):

    if field == "Value":

        return (
            parse_numeric(
                reference_value
            )
            ==
            parse_numeric(
                extracted_value
            )
        )


    if field == "Unit":

        return (
            canonical_null(
                reference_value
            )
            ==
            canonical_null(
                extracted_value
            )
        )


    if field == "Reporting Period":

        return (
            canonical_period(
                reference_value
            )
            ==
            canonical_period(
                extracted_value
            )
        )


    if field == "Source Location":

        return (
            canonical_source_location(
                reference_value
            )
            ==
            canonical_source_location(
                extracted_value
            )
        )


    return (
        normalise_text(
            reference_value
        )
        ==
        normalise_text(
            extracted_value
        )
    )


detailed_rows = []
discrepant_rows = []
missing_rows = []
unsupported_rows = []


all_identities = sorted(

    set(
        reference_index.keys()
    )
    |
    set(
        extraction_index.keys()
    ),

    key=lambda value: (
        value is None,
        str(value)
    )
)


for identity in all_identities:

    ref_entry = (
        reference_index.get(
            identity
        )
    )

    ext_entry = (
        extraction_index.get(
            identity
        )
    )


    if (
        ref_entry is not None
        and ext_entry is None
    ):

        row = (
            ref_entry[
                "record"
            ].copy()
        )

        row[
            "Reference Record Index"
        ] = (
            ref_entry[
                "record_index"
            ]
        )

        missing_rows.append(
            row
        )

        continue


    if (
        ext_entry is not None
        and ref_entry is None
    ):

        row = (
            ext_entry[
                "record"
            ].copy()
        )

        row[
            "Extracted Record Index"
        ] = (
            ext_entry[
                "record_index"
            ]
        )

        unsupported_rows.append(
            row
        )

        continue


    ref_record = (
        ref_entry[
            "record"
        ]
    )

    ext_record = (
        ext_entry[
            "record"
        ]
    )


    field_results = {

        field:

            compare_field(
                field,
                ref_record.get(
                    field
                ),
                ext_record.get(
                    field
                )
            )

        for field
        in EXPECTED_FIELDS
    }


    primary_correct = all(

        field_results[
            field
        ]

        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )


    identity_fields_match = all(

        field_results[
            field
        ]

        for field
        in ALIGNMENT_IDENTITY_FIELDS
    )


    fully_correct = bool(
        primary_correct
    )


    detail = {

        "Reference Record Index":
            ref_entry[
                "record_index"
            ],

        "Extracted Record Index":
            ext_entry[
                "record_index"
            ],

        "Identity Reporting Period":
            identity,

        "Fully Correct":
            fully_correct,

        "Primary Correct":
            bool(
                primary_correct
            ),

        "Alignment Identity Fields Match":
            bool(
                identity_fields_match
            )
    }


    for field in EXPECTED_FIELDS:

        detail[
            f"Reference {field}"
        ] = ref_record.get(
            field
        )

        detail[
            f"Extracted {field}"
        ] = ext_record.get(
            field
        )

        detail[
            f"{field} Correct"
        ] = field_results[
            field
        ]


    detailed_rows.append(
        detail
    )


    if not fully_correct:

        discrepant_rows.append(
            detail.copy()
        )


detailed_df = pd.DataFrame(
    detailed_rows
)

discrepant_df = pd.DataFrame(
    discrepant_rows
)

missing_df = pd.DataFrame(
    missing_rows
)

unsupported_df = pd.DataFrame(
    unsupported_rows
)


fully_correct_df = (
    detailed_df[
        detailed_df[
            "Fully Correct"
        ] == True
    ].copy()
)


print(
    "Aligned:",
    len(
        detailed_df
    )
)

print(
    "Fully correct:",
    len(
        fully_correct_df
    )
)

print(
    "Discrepant:",
    len(
        discrepant_df
    )
)

print(
    "Missing:",
    len(
        missing_df
    )
)

print(
    "Unsupported/unmatched:",
    len(
        unsupported_df
    )
)

In [ ]:
# ============================================================
# 9. Calculate common validation metrics
# ============================================================

N_REF = int(
    len(
        reference_df
    )
)

N_EXT = int(
    len(
        extracted_records
    )
)

N_ALIGNED = int(
    len(
        detailed_df
    )
)

N_CORRECT = int(
    len(
        fully_correct_df
    )
)

N_DISCREPANT = int(
    len(
        discrepant_df
    )
)

N_MISSING = int(
    len(
        missing_df
    )
)

N_UNSUPPORTED = int(
    len(
        unsupported_df
    )
)


completeness = (
    N_ALIGNED
    / N_REF

    if N_REF

    else 0.0
)


missing_rate = (
    N_MISSING
    / N_REF

    if N_REF

    else 0.0
)


record_precision_exact = (
    N_CORRECT
    / N_EXT

    if N_EXT

    else 0.0
)


record_recall_exact = (
    N_CORRECT
    / N_REF

    if N_REF

    else 0.0
)


record_f1_exact = (

    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )

    if (
        record_precision_exact
        + record_recall_exact
    )

    else 0.0
)


unsupported_rate = (
    N_UNSUPPORTED
    / N_EXT

    if N_EXT

    else 0.0
)


discrepancy_rate_among_aligned = (
    N_DISCREPANT
    / N_ALIGNED

    if N_ALIGNED

    else 0.0
)


field_accuracy_among_aligned = {}


for field in EXPECTED_FIELDS:

    if N_ALIGNED:

        field_accuracy_among_aligned[
            field
        ] = float(

            detailed_df[
                f"{field} Correct"
            ].mean()
        )

    else:

        field_accuracy_among_aligned[
            field
        ] = None


correct_field_instances = int(

    sum(

        detailed_df[
            f"{field} Correct"
        ].sum()

        for field
        in PRIMARY_CORRECTNESS_FIELDS
    )
)


expected_field_instances = int(

    N_REF
    * len(
        PRIMARY_CORRECTNESS_FIELDS
    )
)


field_accuracy = (

    correct_field_instances
    / expected_field_instances

    if expected_field_instances

    else 0.0
)


print(
    "Completeness:",
    completeness
)

print(
    "Exact record precision:",
    record_precision_exact
)

print(
    "Exact record recall:",
    record_recall_exact
)

print(
    "Exact record F1:",
    record_f1_exact
)

print(
    "Field accuracy:",
    field_accuracy
)

In [ ]:
# ============================================================
# 10. Field-level diagnostics
# ============================================================

field_accuracy_rows = []


for field in EXPECTED_FIELDS:

    correct_count = int(

        detailed_df[
            f"{field} Correct"
        ].sum()
    )


    field_accuracy_rows.append({

        "Field":
            field,

        "used_in_alignment_identity":
            field
            in ALIGNMENT_IDENTITY_FIELDS,

        "used_in_primary_correctness":
            field
            in PRIMARY_CORRECTNESS_FIELDS,

        "Correct Records":
            correct_count,

        "Aligned Records":
            N_ALIGNED,

        "Accuracy Among Aligned":
            (
                correct_count
                / N_ALIGNED

                if N_ALIGNED

                else None
            ),

        "Overall Accuracy Against Reference":
            (
                correct_count
                / N_REF

                if N_REF

                else None
            )
    })


field_accuracy_df = pd.DataFrame(
    field_accuracy_rows
)


display(
    field_accuracy_df
)

In [ ]:
# ============================================================
# 11. Category-level metrics
# ============================================================

category_metrics = {}


for category in sorted(
    set(
        reference_df[
            "Category"
        ].astype(str)
    )
):

    ref_category_count = int(

        (
            reference_df[
                "Category"
            ]
            == category
        ).sum()
    )


    ext_category_count = int(

        sum(

            1

            for record
            in extracted_records

            if (
                isinstance(
                    record,
                    dict
                )

                and record.get(
                    "Category"
                )
                == category
            )
        )
    )


    if N_ALIGNED:

        aligned_category = (

            detailed_df[
                detailed_df[
                    "Reference Category"
                ]
                == category
            ]
        )

    else:

        aligned_category = (
            pd.DataFrame()
        )


    aligned_category_count = (
        len(
            aligned_category
        )
    )


    fully_correct_category = (

        int(
            aligned_category[
                "Fully Correct"
            ].sum()
        )

        if aligned_category_count

        else 0
    )


    category_precision = (

        fully_correct_category
        / ext_category_count

        if ext_category_count

        else 0.0
    )


    category_recall = (

        fully_correct_category
        / ref_category_count

        if ref_category_count

        else 0.0
    )


    category_f1 = (

        2
        * category_precision
        * category_recall
        / (
            category_precision
            + category_recall
        )

        if (
            category_precision
            + category_recall
        )

        else 0.0
    )


    category_metrics[
        category
    ] = {

        "expected_records":
            ref_category_count,

        "extracted_records":
            ext_category_count,

        "aligned_records":
            aligned_category_count,

        "fully_correct_records":
            fully_correct_category,

        "discrepant_records":
            (
                aligned_category_count
                - fully_correct_category
            ),

        "completeness":
            (
                aligned_category_count
                / ref_category_count

                if ref_category_count

                else 0.0
            ),

        "record_precision_exact":
            category_precision,

        "record_recall_exact":
            category_recall,

        "record_f1_exact":
            category_f1
    }


print(
    json.dumps(
        category_metrics,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 12. Build final Branch B validation summary
# ============================================================

content_diagnostics = {

    "reference_record_count_valid":
        bool(
            reference_record_count_valid
        ),

    "reference_category_counts_valid":
        bool(
            reference_category_counts_valid
        ),

    "extraction_record_count_valid":
        bool(
            N_EXT
            == EXPECTED_RECORD_COUNT
        ),

    "extraction_category_counts_valid":
        bool(
            extracted_df[
                "Category"
            ]
            .value_counts()
            .to_dict()
            == EXPECTED_CATEGORY_COUNTS
        ),

    "branch_B_scope_complete":
        technical_diagnostics.get(
            "scope_complete"
        ),

    "branch_B_content_diagnostics":
        technical_diagnostics.get(
            "content_diagnostics"
        ),

    "reference_identity_unique":
        bool(
            reference_identity_unique
        ),

    "extraction_duplicate_identity_count":
        int(
            len(
                extraction_duplicates
            )
        ),

    "ambiguous_identity_group_count":
        int(
            len(
                alignment_issues
            )
        )
}


comparison_rules = {

    "raw_extraction_modified":
        False,

    "manual_correction_applied":
        False,

    "comparison_normalisation_scope":
        "Comparison copies only",

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "value":
        (
            "Exact represented numeric equality "
            "after deterministic parsing; "
            "no tolerance, rescaling, interpolation, "
            "rounding or conversion."
        ),

    "unit":
        (
            "Exact null agreement. No unit is inferred "
            "because the source CSV contains no explicit "
            "measurement-unit field."
        ),

    "text_fields":
        (
            "Conservative normalised exact agreement "
            "for Category, Topic and Description."
        ),

    "source_location":
        (
            "Exact physical CSV data-row agreement "
            "after deterministic canonicalisation."
        ),

    "equivalence_rules_frozen":
        True
}


matching_rules = {

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "one_to_one_assignment":
        (
            "Unique deterministic Reporting Period "
            "(year) identity."
        ),

    "value_used_for_alignment":
        False,

    "unit_used_for_alignment":
        False,

    "category_used_for_alignment":
        False,

    "topic_used_for_alignment":
        False,

    "description_used_for_alignment":
        False,

    "source_location_used_for_alignment":
        False,

    "matching_rules_frozen_from_branch_A":
        True
}


summary = {

    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "reference_records":
        N_REF,

    "extracted_records":
        N_EXT,

    "aligned_records":
        N_ALIGNED,

    "fully_correct_records":
        N_CORRECT,

    "discrepant_records":
        N_DISCREPANT,

    "missing_records":
        N_MISSING,

    "unsupported_extracted_records":
        N_UNSUPPORTED,

    "completeness":
        round(
            completeness,
            4
        ),

    "missing_rate":
        round(
            missing_rate,
            4
        ),

    "record_precision_exact":
        round(
            record_precision_exact,
            4
        ),

    "record_recall_exact":
        round(
            record_recall_exact,
            4
        ),

    "record_f1_exact":
        round(
            record_f1_exact,
            4
        ),

    "unsupported_rate":
        round(
            unsupported_rate,
            4
        ),

    "discrepancy_rate_among_aligned":
        round(
            discrepancy_rate_among_aligned,
            4
        ),

    "field_accuracy":
        round(
            field_accuracy,
            4
        ),

    "field_accuracy_among_aligned": {

        field:
            (
                round(
                    value,
                    4
                )

                if value is not None

                else None
            )

        for field, value
        in field_accuracy_among_aligned.items()
    },

    "schema_validity":
        schema_validity,

    "schema_diagnostics":
        schema_diagnostics,

    "structurally_evaluable":
        structurally_evaluable,

    "content_diagnostics":
        content_diagnostics,

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "matching_rules":
        matching_rules,

    "comparison_rules":
        comparison_rules,

    "reference_integrity_confirmation": {

        "reference_semantics_valid":
            bool(
                reference_semantics_valid
            ),

        "checks":
            reference_semantic_checks,

        "reference_modified_by_validation":
            False
    },

    "category_metrics":
        category_metrics,

    "input_provenance":
        input_provenance,

    "comparison_rules_frozen_from_branch_A":
        True
}


print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
# ============================================================
# 13. Validation integrity checks
# ============================================================

assert (
    N_ALIGNED
    + N_MISSING
    == N_REF
)


assert (
    N_ALIGNED
    + N_UNSUPPORTED
    == N_EXT
)


assert (
    N_CORRECT
    + N_DISCREPANT
    == N_ALIGNED
)


for metric_name, metric_value in {

    "completeness":
        completeness,

    "missing_rate":
        missing_rate,

    "record_precision_exact":
        record_precision_exact,

    "record_recall_exact":
        record_recall_exact,

    "record_f1_exact":
        record_f1_exact,

    "unsupported_rate":
        unsupported_rate,

    "discrepancy_rate":
        discrepancy_rate_among_aligned,

    "field_accuracy":
        field_accuracy

}.items():

    assert (
        0.0
        <= metric_value
        <= 1.0
    ), (
        f"Invalid "
        f"{metric_name}: "
        f"{metric_value}"
    )


print(
    "Validation integrity checks passed."
)

In [ ]:
# ============================================================
# 14. Export validation artefacts
# ============================================================

DETAILED_PATH = (
    OUTPUT_DIR
    / "D12_branch_B_validation_detailed.csv"
)

FULLY_CORRECT_PATH = (
    OUTPUT_DIR
    / "D12_branch_B_fully_correct_records.csv"
)

DISCREPANT_PATH = (
    OUTPUT_DIR
    / "D12_branch_B_discrepant_records.csv"
)

MISSING_PATH = (
    OUTPUT_DIR
    / "D12_branch_B_missing_records.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR
    / "D12_branch_B_unsupported_records.csv"
)

FIELD_ACCURACY_PATH = (
    OUTPUT_DIR
    / "D12_branch_B_field_accuracy.csv"
)

ALIGNMENT_ISSUES_PATH = (
    OUTPUT_DIR
    / "D12_branch_B_alignment_issues.json"
)

REFERENCE_SEMANTICS_PATH = (
    OUTPUT_DIR
    / "D12_reference_semantics_confirmation.json"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "D12_branch_B_validation_summary.json"
)


detailed_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig"
)


fully_correct_df.to_csv(
    FULLY_CORRECT_PATH,
    index=False,
    encoding="utf-8-sig"
)


discrepant_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig"
)


missing_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig"
)


unsupported_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig"
)


field_accuracy_df.to_csv(
    FIELD_ACCURACY_PATH,
    index=False,
    encoding="utf-8-sig"
)


with open(
    ALIGNMENT_ISSUES_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        alignment_issues,
        f,
        indent=2,
        ensure_ascii=False
    )


with open(
    REFERENCE_SEMANTICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "document_id":
                DOCUMENT_ID,

            "reference_semantics_valid":
                bool(
                    reference_semantics_valid
                ),

            "checks":
                reference_semantic_checks
        },
        f,
        indent=2,
        ensure_ascii=False
    )


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    "D12 Validation B artefacts saved."
)

In [ ]:
# ============================================================
# 15. Download generated validation artefacts
# ============================================================

GENERATED_OUTPUTS = [
    DETAILED_PATH,
    FULLY_CORRECT_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    FIELD_ACCURACY_PATH,
    ALIGNMENT_ISSUES_PATH,
    REFERENCE_SEMANTICS_PATH,
    SUMMARY_PATH
]


for output_path in GENERATED_OUTPUTS:

    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )


for output_path in GENERATED_OUTPUTS:

    if output_path.exists():

        files.download(
            output_path
        )